In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

SRC  = "supplychainiq.bronze.pos_events"
TGT  = "supplychainiq.silver.pos_events"
QUAR = "supplychainiq.silver.pos_quarantine"
CKPT = "/Volumes/supplychainiq/ops/checkpoints/silver_pos"

cols = """event_id STRING, event_ts TIMESTAMP, event_date DATE, store_id STRING, item_id STRING,
          units INT, unit_price DECIMAL(10,2), arrival_date DATE, is_late BOOLEAN,
          _source_file STRING, _ingested_at TIMESTAMP"""
spark.sql(f"CREATE TABLE IF NOT EXISTS {TGT} ({cols}, _silver_at TIMESTAMP) CLUSTER BY (event_date, store_id)")
spark.sql(f"CREATE TABLE IF NOT EXISTS {QUAR} ({cols}, _rescued_data STRING, _dq_reason STRING, _quarantined_at TIMESTAMP)")

In [0]:
def clean(df):
    arrival = F.to_date(F.regexp_extract("_source_file", r"pos/(\d{4}-\d{2}-\d{2})/", 1))
    out = df.select(
        "event_id",
        F.to_timestamp("event_ts").alias("event_ts"),
        F.to_date("event_date").alias("event_date"),
        "store_id", "item_id",
        F.col("units").cast("int").alias("units"),
        F.col("unit_price").cast("decimal(10,2)").alias("unit_price"),
        arrival.alias("arrival_date"),
        "_source_file", "_ingested_at", "_rescued_data",
    ).withColumn("is_late", F.col("arrival_date") > F.col("event_date"))
    reason = (F.when(F.col("event_id").isNull(), "missing_event_id")
               .when(F.col("_rescued_data").isNotNull(), "rescued_data")
               .when(F.col("event_date").isNull(), "invalid_event_date")
               .when(F.col("units").isNull() | (F.col("units") <= 0), "invalid_units"))
    return out.withColumn("_dq_reason", reason)

def process(batch_df, batch_id):
    s = batch_df.sparkSession
    df = clean(batch_df)

    # 1) bad rows -> quarantine (txnVersion makes a retried batch a no-op)
    bad = df.filter("_dq_reason IS NOT NULL").withColumn("_quarantined_at", F.current_timestamp())
    (bad.select(*s.table(QUAR).columns).write.mode("append")
        .option("txnAppId", "silver_pos_quarantine").option("txnVersion", batch_id)
        .saveAsTable(QUAR))

    # 2) dedup inside the batch: keep the copy that arrived FIRST
    first = Window.partitionBy("event_id").orderBy("arrival_date", "_source_file")
    good = (df.filter("_dq_reason IS NULL")
              .withColumn("_rn", F.row_number().over(first)).filter("_rn = 1")
              .withColumn("_silver_at", F.current_timestamp())
              .select(*s.table(TGT).columns))

    # 3) dedup across batches: insert only events Silver has never seen
    (DeltaTable.forName(s, TGT).alias("t")
        .merge(good.alias("s"), "t.event_date = s.event_date AND t.event_id = s.event_id")
        .whenNotMatchedInsertAll()
        .execute())

before = spark.table(TGT).count()
q = (spark.readStream.table(SRC)
       .writeStream.foreachBatch(process)
       .option("checkpointLocation", CKPT)
       .trigger(availableNow=True)
       .start())
q.awaitTermination()
after = spark.table(TGT).count()
print(f"{after - before:,} new events in Silver (total {after:,})")

In [0]:
%sql
WITH b AS (SELECT COUNT(*) n, COUNT(DISTINCT event_id) d FROM supplychainiq.bronze.pos_events),
     s AS (SELECT COUNT(*) n, COUNT(DISTINCT event_id) d FROM supplychainiq.silver.pos_events),
     t AS (SELECT COUNT(*) n FROM supplychainiq.ops.fault_truth
           WHERE fault_type = 'duplicate_pos_event'
             AND regexp_extract(detail, 'resent_in=([0-9-]+)', 1) <= '2015-08-20')
SELECT b.n AS bronze_rows, b.n - b.d AS bronze_dups, t.n AS truth_dups,
       s.n AS silver_rows, s.n - s.d AS silver_dups,
       (SELECT COUNT(*) FROM supplychainiq.silver.pos_quarantine) AS quarantined
FROM b, s, t;

In [0]:
%sql
SELECT event_date, store_id, COUNT(*) AS late_events
FROM supplychainiq.silver.pos_events
WHERE is_late
GROUP BY ALL HAVING COUNT(*) > 50
ORDER BY event_date;